# Visualização das Anotações nas Imagens de Mergulho

Autor:  Viviane da Rosa Sommer

Atualizado: 23/01/2025

## Objetivo:
Este notebook tem como objetivo mostrar as imagens originais de ergulho, juntamente com sua anotação de Coral-Sol.

## Importações Necessárias

In [ ]:
import os 
import matplotlib.pyplot as plt 
from PIL import Image 
import numpy as np 
from matplotlib import patches 
import traceback 

## Funções Necessárias

In [ ]:
def plot_image_with_annotations(image_path: str, annotation_path: str) -> None: 
    """ 
    Plot the original image and the image with annotations side by side. 
 
    Args: 
        image_path (str): Path to the image file. 
        annotation_path (str): Path to the annotation file. 
 
    Returns: 
        None 
    """ 
    try: 
        image = Image.open(image_path) 
         
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10)) 
         
        ax1.imshow(image) 
        ax1.axis('off') 
        ax1.set_title('Original Image') 
         
        ax2.imshow(image) 
        ax2.axis('off') 
        ax2.set_title('Image with Annotations') 
 
        annotations = load_annotations(annotation_path, image.size) 
        if not annotations: 
            print(f"No annotations found for {image_path}") 
            return 
 
        for annotation in annotations: 
            class_id, polygon = annotation 
            patch = patches.Polygon(polygon, closed=True, edgecolor='r', facecolor='none', lw=2) 
            ax2.add_patch(patch) 
 
        main_title = os.path.splitext(os.path.basename(image_path))[0] 
        plt.suptitle(main_title, fontsize=16) 
 
        plt.tight_layout() 
        plt.show() 
 
    except Exception as e: 
        print(f"Error processing annotation for {image_path}: {e}") 
        traceback.print_exc() 
 
def load_annotations(annotation_path: str, image_size: tuple) -> list: 
    """ 
    Load annotations from a file and denormalize coordinates. 
 
    Args: 
        annotation_path (str): Path to the annotation file. 
        image_size (tuple): Size of the image (width, height). 
 
    Returns: 
        list: List of tuples containing class_id and polygon points. 
    """ 
    try: 
        annotations = [] 
        with open(annotation_path, 'r') as file: 
            for line in file: 
                parts = line.strip().split() 
                class_id = int(parts[0]) 
                points = np.array(list(map(float, parts[1:]))) 
                points = points.reshape(-1, 2) 
 
                points[:, 0] *= image_size[0] 
                points[:, 1] *= image_size[1] 
 
                annotations.append((class_id, points)) 
 
        return annotations 
    except Exception as e: 
        print(f"Error reading annotation file {annotation_path}: {e}") 
        traceback.print_exc() 
        return [] 
 
def process_dataset_images(dataset_folder: str) -> None: 
    """ 
    Process all images in the dataset folder and plot them with annotations. 
 
    Args: 
        dataset_folder (str): Path to the folder containing the images. 
 
    Returns: 
        None 
    """ 
    for image_filename in os.listdir(dataset_folder): 
        try: 
            if image_filename.lower().endswith(('.png', '.jpg', '.jpeg')): 
                image_path = os.path.join(dataset_folder, image_filename) 
                annotation_path = get_annotation_path(dataset_folder, image_filename) 
 
                if os.path.exists(annotation_path): 
                    plot_image_with_annotations(image_path, annotation_path) 
                else: 
                    print(f"No corresponding annotation file for {image_filename}") 
        except Exception as e: 
            print(f"Error processing file {image_filename}: {e}") 
            traceback.print_exc() 
 
def get_annotation_path(dataset_folder: str, image_filename: str) -> str: 
    """ 
    Get the path to the annotation file corresponding to an image. 
 
    Args: 
        dataset_folder (str): Path to the folder containing the images. 
        image_filename (str): Name of the image file. 
 
    Returns: 
        str: Path to the corresponding annotation file. 
    """ 
    base_filename = os.path.splitext(image_filename)[0] 
    return os.path.join(dataset_folder, f"{base_filename}.txt").replace("original_images","yolo_annotation") 

## Visualização das imagens e suas anotações

In [ ]:
dataset_folder = 'Dataset-Segm-Mergulho/original_images/' 
process_dataset_images(dataset_folder) 

In [ ]:
!jupyter nbconvert --to html Visualizacao_Anotacao_Mergulho.ipynb